In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

df = pd.read_csv('../data/cars_cleaned.csv')

X = df.drop(columns=['Price', 'log_price'])
y = df['log_price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

preprocessor = joblib.load('../src/preprocessor.pkl')

In [6]:
import mlflow
from sklearn.linear_model import LinearRegression
from lightgbm import LGBMRegressor
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

models = {
    'linear_regression': LinearRegression(),
    'lightgbm': LGBMRegressor(n_estimators=300, random_state=42)
}

results = {}

for name, model in models.items():
    with mlflow.start_run(run_name=name):
        pipeline = Pipeline([
            ('preprocessor', preprocessor),
            ('model', model)
        ])

        pipeline.fit(X_train, y_train)
        preds_log = pipeline.predict(X_test)

        # Evaluate in real MAD, not log-space — this is what actually matters for the business problem
        preds_price = np.expm1(preds_log)
        y_test_price = np.expm1(y_test)

        rmse = root_mean_squared_error(y_test_price, preds_price)
        mae = mean_absolute_error(y_test_price, preds_price)
        r2 = r2_score(y_test_price, preds_price)

        mlflow.log_param('model_type', name)
        mlflow.log_metrics({'rmse_mad': rmse, 'mae_mad': mae, 'r2': r2})
        mlflow.sklearn.log_model(pipeline, name)

        results[name] = {'rmse': rmse, 'mae': mae, 'r2': r2, 'pipeline': pipeline}
        print(f"{name}: RMSE={rmse:.0f} MAD, MAE={mae:.0f} MAD, R²={r2:.3f}")

2026/06/17 05:39:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/17 05:39:08 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


linear_regression: RMSE=94160 MAD, MAE=39139 MAD, R²=0.489
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005254 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 608
[LightGBM] [Info] Number of data points in the train set: 60368, number of used features: 30
[LightGBM] [Info] Start training from score 11.556147


c:\Users\Dell\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
2026/06/17 05:39:14 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/17 05:39:14 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


lightgbm: RMSE=69246 MAD, MAE=22277 MAD, R²=0.724


In [7]:
best_pipeline = results['lightgbm']['pipeline']
joblib.dump(best_pipeline, '../src/model.pkl')

['../src/model.pkl']